# 02 Clean + time split

**Input:** `data/raw/CCDB_Export_9_January_2025_through_February_2025.zip` (never edited).
**Output:** `data/processed/complaints_clean.csv.gz` (one row per distinct cleaned narrative, with a `split` column) and `reports/cleaning_log.csv` (row counts after every step).

Decisions, in the order the code applies them:
1. Keep only rows with a narrative.
2. Clean the text: remove redaction (`XXXX`, `XX/XX/XXXX`, `XX/XX/year>`), remove the braces around amounts (`{$500.00}` -> `$500.00`), squeeze extra spaces. Keep the original case (TF-IDF lowercases later).
3. Drop narratives that are empty after cleaning.
4. Drop texts that appear under more than one product. Same words, different labels: we can't know which label is right.
5. Remove exact duplicates of the CLEANED text, keeping the earliest copy. Cleaning first matters: two texts that only differed in their redaction become identical.
6. Split by time: train = Jan 1-24, validation = Jan 25-31, test = February.
7. Rare products: fold any product with fewer than 500 texts in training into "Other" (checked after the split, because the rule counts training rows only).
8. Check that no text appears in two splits.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "README.md").exists():
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw" / "CCDB_Export_9_January_2025_through_February_2025.zip"
OUT = ROOT / "data" / "processed" / "complaints_clean.csv.gz"
LOG = ROOT / "reports" / "cleaning_log.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

COLS = ["Date received", "Product", "Sub-product", "Issue",
        "Consumer complaint narrative", "Company", "Complaint ID"]
df = pd.read_csv(RAW, usecols=COLS, dtype=str, keep_default_na=False)
df["Date received"] = pd.to_datetime(df["Date received"], format="%Y-%m-%d")
print(df.shape)

(811019, 7)


## A small log

Every step adds one line: what we did and how many rows are left. At the end this becomes `reports/cleaning_log.csv`, so every number in the write-up traces back to a file.

In [2]:
log = []

def record(step, frame):
    log.append({"step": step, "rows": len(frame)})
    print(f"{step:<45} {len(frame):>8,}")

record("raw rows", df)

raw rows                                       811,019


## Step 1: keep rows with a narrative

In [3]:
df = df[df["Consumer complaint narrative"].str.strip() != ""].copy()
record("1. has a narrative", df)

1. has a narrative                             265,405


## Step 2: clean the text

Three regex replacements (`regex=True` means the pattern is a regular expression):

| Pattern | Means | Replaced with |
|---|---|---|
| `X{2,}` | 2 or more capital X in a row | a space |
| `year>` | a leftover piece of the CFPB's `XX/XX/year>` date redaction | a space |
| `[{}]` | a `{` or a `}` character | nothing |
| `\s+` | one or more whitespace characters (spaces, newlines, tabs) | one space |

Why "2 or more X" and not exactly `XXXX`: notebook 01 showed redaction glued to other letters (`CXXXX`, `XXXXAccount`) and runs of 6 or 8 X. Known side effect: a real word with "XX" in capitals (for example "EXXON") loses its X's. That is rare and costs little.

Leftover slashes from `XX/XX/XXXX` stay as `/ /`; TF-IDF ignores punctuation, so they do no harm.

In [4]:
def clean_text(s: pd.Series) -> pd.Series:
    s = s.str.replace(r"X{2,}", " ", regex=True)
    s = s.str.replace("year>", " ", regex=False)
    s = s.str.replace(r"[{}]", "", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

# Try it on a made-up example first, so we can see exactly what each rule does.
demo = pd.Series(["On XX/XX/XXXX I paid {$500.00} to XXXX XXXX. Account XXXXXXXX1234,  closed XX/XX/year>."])
print(demo[0])
print(clean_text(demo)[0])

On XX/XX/XXXX I paid {$500.00} to XXXX XXXX. Account XXXXXXXX1234,  closed XX/XX/year>.
On / / I paid $500.00 to . Account 1234, closed / / .


In [5]:
df["text"] = clean_text(df["Consumer complaint narrative"])

# Check: how many tokens with XX survive? (should be 0)
print("XX left:", df["text"].str.contains("XX", regex=False).sum())

# Before / after on one real narrative
i = df.index[0]
print("\nBEFORE:", df.at[i, "Consumer complaint narrative"][:300])
print("\nAFTER: ", df.at[i, "text"][:300])

XX left: 0

BEFORE: I am a victim of identity theft and I write to dispute certain information in my file resulting from the crime. The items do not relate to any transactions I have made or have authorized. Please remove/correct this information at the earliest possible time. As required by section 611 of the fair cre

AFTER:  I am a victim of identity theft and I write to dispute certain information in my file resulting from the crime. The items do not relate to any transactions I have made or have authorized. Please remove/correct this information at the earliest possible time. As required by section 611 of the fair cre


## Step 3: drop texts that are empty after cleaning

A narrative that was only redaction (for example `XXXX XXXX`) becomes empty.

In [6]:
print("empty after cleaning:", (df["text"] == "").sum())
df = df[df["text"] != ""]
record("3. not empty after cleaning", df)

empty after cleaning: 13
3. not empty after cleaning                    265,392


## Step 4: drop texts with conflicting labels

`groupby("text")["Product"].nunique()` gives, for every distinct text, how many different products it was filed under. More than 1 means the same words carry different labels.

In [7]:
n_products = df.groupby("text")["Product"].nunique()
conflicting = n_products[n_products > 1].index

print("texts with more than one product:", len(conflicting))
print("rows they cover:                 ", df["text"].isin(conflicting).sum())

# The biggest ones: how many copies, and which products
(df[df["text"].isin(conflicting)]
   .groupby("text")["Product"].agg(copies="size", products=lambda s: sorted(s.unique()))
   .sort_values("copies", ascending=False)
   .head(5)
   .rename(index=lambda t: t[:70] + "..."))

texts with more than one product: 175
rows they cover:                  25507


,copies,products
text,,
"I am filing a complaint against Cash App ( Block , Inc. ) due to inade...",9269,"[Checking or savings account, Credit reporting..."
I am writing to formally express my concerns regarding Zelle 's handli...,4113,"[Checking or savings account, Credit reporting..."
I am writing to file a complaint against Navy Federal Credit Union reg...,2382,"[Checking or savings account, Credit card]"
Could you please take a moment to review and examine the attached docu...,1434,[Credit reporting or other personal consumer r...
Experian has failed to fulfill its legal obligation under the Fair Cre...,1215,[Credit reporting or other personal consumer r...


In [8]:
df = df[~df["text"].isin(conflicting)]
record("4. drop conflicting labels", df)

4. drop conflicting labels                     239,885


## Step 5: exact dedupe on the cleaned text, keep the earliest

Sort by date first, so `keep="first"` keeps the earliest copy. `Complaint ID` is a tie-breaker for copies on the same day, so the result is the same every time we run it.

Why the earliest: if a text appears in January and again in February, the copy we keep is in January (train). February (test) then only holds texts the model has never seen.

In [9]:
print("distinct raw narratives:    ", df["Consumer complaint narrative"].nunique())
print("distinct cleaned narratives:", df["text"].nunique())

df = (df.sort_values(["Date received", "Complaint ID"])
        .drop_duplicates(subset="text", keep="first"))
record("5. dedupe cleaned text (keep earliest)", df)

distinct raw narratives:     147206
distinct cleaned narratives: 138042
5. dedupe cleaned text (keep earliest)         138,042


## Step 6: time split

| Split | Dates | Used for |
|---|---|---|
| train | Jan 1-24, 2025 | fitting models |
| valid | Jan 25-31, 2025 | choosing the triage threshold (notebook 05) |
| test | Feb 1-28, 2025 | the final score, used once |

`pd.cut` would also work, but a nested `where` is easier to read here.

In [10]:
import numpy as np

d = df["Date received"]
df["split"] = np.where(d < "2025-01-25", "train",
              np.where(d < "2025-02-01", "valid", "test"))

counts = df["split"].value_counts().reindex(["train", "valid", "test"])
print(counts)
print(df.groupby("split")["Date received"].agg(["min", "max"]).reindex(["train", "valid", "test"]))

split
train    77504
valid    18093
test     42445
Name: count, dtype: int64
             min        max
split                      
train 2025-01-01 2025-01-24
valid 2025-01-25 2025-01-31
test  2025-02-01 2025-02-28


## Step 7: rare products

Rule: a product with fewer than 500 texts in TRAIN is folded into "Other". 500 is a judgement call: with fewer, the per-class scores on validation and test rest on a handful of complaints and swing a lot.

In [11]:
MIN_TRAIN = 500
train_counts = df.loc[df["split"] == "train", "Product"].value_counts()
rare = train_counts[train_counts < MIN_TRAIN].index.tolist()
print("products below", MIN_TRAIN, "in train:", rare)

if rare:
    df["Product"] = df["Product"].where(~df["Product"].isin(rare), "Other")
print(df["Product"].nunique(), "products")

products below 500 in train: ['Debt or credit management', 'Prepaid card']
10 products


In [12]:
# Product share per split: is February's mix similar to January's?
mix = pd.crosstab(df["Product"], df["split"], normalize="columns")[["train", "valid", "test"]]
mix.round(3).sort_values("train", ascending=False)

split,train,valid,test
Product,,,
Credit reporting or other personal consumer reports,0.491,0.639,0.583
"Money transfer, virtual currency, or money service",0.235,0.082,0.068
Checking or savings account,0.112,0.068,0.073
Debt collection,0.072,0.106,0.123
Credit card,0.037,0.045,0.064
Mortgage,0.014,0.015,0.024
Student loan,0.012,0.013,0.023
Vehicle loan or lease,0.011,0.013,0.019
Other,0.009,0.010,0.012


In [13]:
# Counts per product per split (the small classes matter most: how many test examples do they have?)
pd.crosstab(df["Product"], df["split"])[["train", "valid", "test"]].sort_values("train", ascending=False)

split,train,valid,test
Product,,,
Credit reporting or other personal consumer reports,38079,11566,24740
"Money transfer, virtual currency, or money service",18207,1488,2906
Checking or savings account,8666,1237,3101
Debt collection,5581,1920,5240
Credit card,2879,818,2710
Mortgage,1055,270,1005
Student loan,968,234,964
Vehicle loan or lease,852,238,802
Other,700,177,526


## Step 8: leakage check

Each set below holds the texts of one split. `&` on two sets gives the texts they share. All three overlaps must be 0. `assert` stops the notebook with an error if not.

In [14]:
texts = {s: set(df.loc[df["split"] == s, "text"]) for s in ["train", "valid", "test"]}
overlaps = {
    "train & valid": len(texts["train"] & texts["valid"]),
    "train & test":  len(texts["train"] & texts["test"]),
    "valid & test":  len(texts["valid"] & texts["test"]),
}
print(overlaps)
assert all(v == 0 for v in overlaps.values())
assert df["Complaint ID"].is_unique

{'train & valid': 0, 'train & test': 0, 'valid & test': 0}


## Save

In [15]:
keep = ["Complaint ID", "Date received", "Product", "Sub-product", "Issue", "Company", "text", "split"]
df[keep].to_csv(OUT, index=False)
print("saved", OUT.name, len(df), "rows,", round(OUT.stat().st_size / 1e6, 1), "MB")

for s in ["train", "valid", "test"]:
    log.append({"step": f"split: {s}", "rows": int((df["split"] == s).sum())})
pd.DataFrame(log).to_csv(LOG, index=False)
pd.DataFrame(log)

saved complaints_clean.csv.gz 138042 rows, 38.8 MB


,step,rows
0,raw rows,811019
1,1. has a narrative,265405
2,3. not empty after cleaning,265392
3,4. drop conflicting labels,239885
4,5. dedupe cleaned text (keep earliest),138042
5,split: train,77504
6,split: valid,18093
7,split: test,42445
